In [0]:
WITH sku_usage_table AS (
  SELECT
    sku_name,
    usage_date AS date,
    count(distinct identity_metadata.run_as) AS number_of_users
  FROM system.billing.usage
  WHERE year(usage_date) = 2026
    AND (sku_name = 'PREMIUM_ALL_PURPOSE_SERVERLESS_COMPUTE_US_EAST_2'
         OR sku_name = 'PREMIUM_JOBS_SERVERLESS_COMPUTE_US_EAST_2')
  GROUP BY usage_date, sku_name
),
total_usage AS (
  SELECT
    date,
    sum(number_of_users) AS total_number_of_users
  FROM sku_usage_table
  GROUP BY date
)
SELECT
  s.sku_name,
  s.date,
  s.number_of_users,
  s.number_of_users / t.total_number_of_users AS pct_of_total_users
FROM sku_usage_table s
INNER JOIN total_usage t
  ON s.date = t.date
ORDER BY s.date